In [1]:
import numpy as np
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
import geoplotlib
from pathlib import Path

def data_file(filename):
    candidates = [Path('../Datasets') / filename, Path('Datasets') / filename, Path(filename)]
    for path in candidates:
        if path.is_file():
            return str(path)
    raise FileNotFoundError(f"Place {filename} in ../Datasets, Datasets, or this notebook folder")
import pandas as pd
import pyglet
from geoplotlib.utils import DataAccessObject, BoundingBox
from geoplotlib.layers import BaseLayer
from geoplotlib.core import BatchPainter

In [2]:
dataset = pd.read_csv(data_file('world_cities_pop.csv'), dtype={'Region': str}, low_memory=False)
dataset.head()

,Country,City,AccentCity,Region,Population,Latitude,Longitude
0,ad,aixas,Aixàs,06,NaN,42.483333,1.466667
1,ad,aixirivali,Aixirivali,06,NaN,42.466667,1.500000
2,ad,aixirivall,Aixirivall,06,NaN,42.466667,1.500000
3,ad,aixirvall,Aixirvall,06,NaN,42.466667,1.500000
4,ad,aixovall,Aixovall,06,NaN,42.466667,1.483333


In [3]:
dataset = dataset.rename(columns={'Latitude': 'lat', 'Longitude': 'lon'})
dataset['lat'] = pd.to_numeric(dataset['lat'], errors='coerce')
dataset['lon'] = pd.to_numeric(dataset['lon'], errors='coerce')
dataset = dataset.dropna(subset=['lat', 'lon']).copy()
dataset = dataset[dataset['lat'].between(-90, 90) & dataset['lon'].between(-180, 180)].copy()
dataset['Country'] = dataset['Country'].astype('string').str.lower().str.strip()
dataset.head()

,Country,City,AccentCity,Region,Population,lat,lon
0,ad,aixas,Aixàs,06,NaN,42.483333,1.466667
1,ad,aixirivali,Aixirivali,06,NaN,42.466667,1.500000
2,ad,aixirivall,Aixirivall,06,NaN,42.466667,1.500000
3,ad,aixirvall,Aixirvall,06,NaN,42.466667,1.500000
4,ad,aixovall,Aixovall,06,NaN,42.466667,1.483333


In [4]:
europe_country_codes = [
    'al', 'ad', 'at', 'by', 'be', 'ba', 'bg', 'hr', 'cy', 'cz', 'dk', 'ee',
    'fi', 'fr', 'de', 'gr', 'hu', 'is', 'ie', 'it', 'lv', 'lt', 'lu', 'mt',
    'md', 'mc', 'me', 'nl', 'mk', 'no', 'pl', 'pt', 'ro', 'ru', 'sm', 'rs',
    'sk', 'si', 'es', 'se', 'ch', 'ua', 'gb', 'uk', 'va'
]
europe_data = dataset[dataset['Country'].isin(europe_country_codes)].copy()

In [5]:
print('All cities:', len(dataset))
print('European cities:', len(europe_data))

All cities: 3173958
European cities: 897761


In [6]:
name_column = 'AccentCity' if 'AccentCity' in europe_data.columns else 'City'
cities_z = europe_data[europe_data[name_column].astype('string').str.upper().str.startswith('Z', na=False)].copy()

In [7]:
print('European cities starting with Z:', len(cities_z))
cities_z.head()

European cities starting with Z: 20738


,Country,City,AccentCity,Region,Population,lat,lon
104206,al,zaane,ZÄane,44,NaN,40.932778,19.783056
104207,al,zabarzani,Zabarzani,40,NaN,40.427778,20.269167
104208,al,zabarzan,Zabarzan,40,NaN,40.427778,20.269167
104209,al,zaberzane,Zabërzanë,40,NaN,40.427778,20.269167
104210,al,zaberzan i siperm,Zaberzan i Sipërm,40,NaN,40.427778,20.269167


In [8]:
z_geo = DataAccessObject(cities_z)
geoplotlib.dot(z_geo, f_tooltip=lambda row: str(row['Country']) + ' - ' + str(row[name_column]))
geoplotlib.show()

In [9]:
geoplotlib.voronoi(z_geo, cmap='Reds_r', alpha=50)
geoplotlib.show()

In [12]:
start_letters = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

class FilterLayer(BaseLayer):
    def __init__(self, data, bbox=BoundingBox.WORLD):
        self.data = data
        self.view = bbox
        self.start_letter = 0
        self.painter = BatchPainter()
        self.name_column = ('AccentCity' if 'AccentCity' in data.keys() else 'City')

    def invalidate(self, proj):
        self.filtered = self.data.where(
            np.array([str(city).upper().startswith(start_letters[self.start_letter])
                      for city in self.data[self.name_column]])
        )

    def draw(self, proj, mouse_x, mouse_y, ui_manager):
        self.painter = BatchPainter()
        x, y = proj.lonlat_to_screen(self.filtered['lon'], self.filtered['lat'])
        self.painter.set_color([30, 50, 255, 180])
        self.painter.points(x, y, 2)
        self.painter.batch_draw()
        ui_manager.info('Displaying cities starting with ' + start_letters[self.start_letter])

    def on_key_release(self, key, modifiers):
        if key == pyglet.window.key.RIGHT:
            self.start_letter = (self.start_letter + 1) % 26
            return True
        if key == pyglet.window.key.LEFT:
            self.start_letter = (self.start_letter - 1) % 26
            return True
        return False

    def bbox(self):
        return self.view

In [13]:
europe_bbox = BoundingBox(north=72, west=-12, south=34, east=45)
europe_geo = DataAccessObject(europe_data)
geoplotlib.add_layer(FilterLayer(europe_geo, bbox=europe_bbox))
geoplotlib.show()